[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C24_Inference_Serving_Course/03_speculative/03_speculative.ipynb)

# 03 · 投机解码 Speculative Decoding（用 numpy 做模拟器+验证）

目标：把 **draft-then-verify** 框架、**speculative sampling 接受/拒绝规则**、**无损性（输出分布逐位等于 target）**、**接受率→期望确认 token 数**、**净加速 cost model** 用 numpy 实现并验证。

路线：朴素自回归采样(参考) → 单步投机采样(接受/拒绝+残差) → 残差分布是合法分布 → **无损验证(30万次采样对拍 p_target)** → 期望 token 数对拍闭式 → 净加速 cost model → ✏️ 练习 → 📖 答案 → 🧪 真实接受率胶囊。

> 心智模型：**draft 只影响速度(接受率)，不影响正确性；接受规则 min(1,p_t/p_d) + 残差重采样 把 draft 分布精确掰回 target 分布**。

## 1 · 朴素自回归采样（参考基线）

先写一个最朴素的「逐 token 从 target 分布采样」当**参考**——投机解码的输出分布必须与它**逐位一致**。
玩具设定：词表大小 V，target 是一个固定的概率分布（真实中它来自一次 target 前向的 softmax）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_dist(V, rng):
    p = rng.random(V); p /= p.sum(); return p

def naive_sample(p_target, n, rng):
    '''朴素自回归：逐 token 直接从 target 分布采样（这里各步独立，作分布参考）。'''
    V = len(p_target)
    return rng.choice(V, size=n, p=p_target)

V = 6
p_target = make_dist(V, rng)
samples = naive_sample(p_target, 100000, rng)
emp = np.bincount(samples, minlength=V) / len(samples)
print('p_target:', np.round(p_target, 4))
print('经验频率:', np.round(emp, 4))
assert np.allclose(emp, p_target, atol=0.01)
print('✅ 朴素采样的经验频率 ≈ p_target —— 这就是投机解码必须复现的金标准分布')

## 2 · 单步投机采样：接受/拒绝 + 残差重采样

核心规则：draft 提议 `x ~ q(draft分布)`；以 `min(1, p(x)/q(x))` 接受；拒绝则从残差 `norm(max(0,p−q))` 重采样。
下面实现单步逻辑，返回最终 token 以及是否被接受。

In [ ]:
def residual_dist(p, q):
    '''校正/残差分布：只在 p>q 处有质量，再归一化。'''
    r = np.maximum(p - q, 0.0)
    s = r.sum()
    return r / s if s > 0 else p.copy()

def spec_sample_step(p_target, q_draft, rng):
    '''一步投机采样。返回 (最终token, 是否接受了draft提议)。'''
    x = rng.choice(len(q_draft), p=q_draft)           # draft 提议
    accept_prob = min(1.0, p_target[x] / q_draft[x])  # 接受概率
    if rng.random() < accept_prob:
        return x, True                                # 接受
    resid = residual_dist(p_target, q_draft)          # 拒绝 -> 残差重采样
    return rng.choice(len(q_draft), p=resid), False

q_draft = make_dist(V, rng)
tok, acc = spec_sample_step(p_target, q_draft, rng)
print(f'draft 分布: {np.round(q_draft,3)}')
print(f'一步投机 -> token={tok}, 接受={acc}')
# 接受概率上界检查：min(1, p/q) <= 1
for x in range(V):
    assert min(1.0, p_target[x]/q_draft[x]) <= 1.0
print('✅ 单步投机采样实现完成（接受概率恒 <= 1）')

## 3 · 残差分布是合法概率分布

残差 `norm(max(0,p−q))` 必须非负、和为 1，才是合法分布。再验证一个恒等式（无损证明的核心一步）：
`min(q,p) + max(0,p−q) == p`，逐元素成立。

In [ ]:
for _ in range(200):
    p = make_dist(V, rng); q = make_dist(V, rng)
    r = residual_dist(p, q)
    assert np.all(r >= -1e-12), '残差应非负'
    assert abs(r.sum() - 1.0) < 1e-9, '残差应和为 1'
    # 无损证明的核心恒等式：min(q,p)+max(0,p-q)=p
    assert np.allclose(np.minimum(q, p) + np.maximum(p - q, 0.0), p)
print('✅ 残差分布合法(非负、和为1)；恒等式 min(q,p)+max(0,p−q)=p 处处成立')
print('   这个恒等式正是「接受路径 + 拒绝重采样路径 = p」无损证明的关键。')

## 4 · 🔑 无损验证：输出分布逐位等于 target

这是本模块**最核心的不变量**：无论 draft 多差，投机采样的输出分布都**精确等于** target。
用 30 万次单步投机采样统计经验频率，断言它 `np.allclose` 到 `p_target`。

In [ ]:
def spec_empirical(p_target, q_draft, N, rng):
    V = len(p_target)
    cnt = np.zeros(V)
    for _ in range(N):
        tok, _ = spec_sample_step(p_target, q_draft, rng)
        cnt[tok] += 1
    return cnt / N

# 故意用一个很差的 draft，检验无损性与 draft 质量无关
p_target = make_dist(V, rng)
q_bad = make_dist(V, rng)            # 和 target 毫无关系的烂 draft
emp = spec_empirical(p_target, q_bad, 300000, rng)
print('p_target :', np.round(p_target, 4))
print('投机输出 :', np.round(emp, 4))
print('最大误差 :', f'{np.max(np.abs(emp - p_target)):.4f}')
assert np.allclose(emp, p_target, atol=0.01), '投机输出分布必须等于 target！'
print('✅ 即便用毫不相关的烂 draft，投机采样输出仍精确等于 p_target —— 无损！')
print('   draft 只决定速度(接受率)，绝不改变正确性。这是投机解码能用的根本前提。')

## 5 · 接受率 → 期望确认 token 数（对拍闭式）

一轮 draft-verify 平均确认 `E = (1−α^(γ+1))/(1−α)` 个 token。
用 Bernoulli(α) 模型模拟「逐位接受、遇拒即停、末尾 +1」，统计平均确认数，对拍闭式。

In [ ]:
def expected_tokens_closed(alpha, gamma):
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)

def sim_round_tokens(alpha, gamma, rng):
    '''一轮：γ 个 draft token 逐位以 α 接受，遇第一个拒绝即停；
       末尾恒 +1（拒绝处的校正 token，或全接受时 target 的 bonus token）。'''
    n = 0
    for _ in range(gamma):
        if rng.random() < alpha:
            n += 1
        else:
            break
    return n + 1

N = 200000
print(f"{'alpha':>6}{'gamma':>6}{'模拟':>9}{'闭式':>9}{'误差':>8}")
for alpha, gamma in [(0.5,4),(0.7,4),(0.8,5),(0.9,8)]:
    sim = np.mean([sim_round_tokens(alpha, gamma, rng) for _ in range(N)])
    closed = expected_tokens_closed(alpha, gamma)
    print(f'{alpha:>6}{gamma:>6}{sim:>9.3f}{closed:>9.3f}{abs(sim-closed):>8.3f}')
    assert abs(sim - closed) < 0.03, '模拟应对拍闭式'
print('✅ 期望确认 token 数模拟 == 闭式 (1−α^(γ+1))/(1−α)')
print('   观察：α 越高、γ 越长收益越大，但 γ 的回报随 α^(γ+1) 递减。')

## 6 · 净加速 cost model：把 draft 开销算进来

净加速 `= E / (γ·c + 1)`：分子是每轮确认数，分母是一轮总成本（draft γ 步×c + verify 一次）。
扫不同 γ，找**最优草稿长度**——太短摊不开 verify，太长后段白算。

In [ ]:
def net_speedup(alpha, gamma, c):
    '''target 单步成本=1, draft 单步成本=c。返回相对 target 自回归的加速比。'''
    E = expected_tokens_closed(alpha, gamma)
    cost_per_token = (gamma * c + 1) / E
    return 1.0 / cost_per_token

alpha, c = 0.8, 0.1            # 好 draft，draft 比 target 快 10 倍
print(f'设 α={alpha}, draft 成本 c={c} (target=1)')
print(f"{'gamma':>6}{'加速比':>9}")
best = (0, 0.0)
for gamma in range(1, 11):
    s = net_speedup(alpha, gamma, c)
    star = ' <-- 最优' if s > best[1] else ''
    if s > best[1]: best = (gamma, s)
    print(f'{gamma:>6}{s:>9.3f}{star}')
print(f'\n最优 γ={best[0]}, 加速 {best[1]:.2f}x')
assert best[1] > 1.0, '好 draft 应有净加速'
assert 1 <= best[0] <= 10
# 烂 draft (低 α) 可能净加速 < 1
bad = max(net_speedup(0.3, g, c) for g in range(1, 11))
print(f'对比：α=0.3 的烂 draft 最优加速仅 {bad:.2f}x（接近甚至不值得投机）')
print('✅ 存在最优 γ：太短摊不开 verify 固定成本，太长后段 draft 大概率白算')

---
## ✏️ 练习 1：实现接受/拒绝单步

实现 `accept_or_resample(p_target, q_draft, x, u_accept, rng)`：给定 draft 已提议的 token `x` 和一个用于接受判定的随机数 `u_accept∈[0,1)`，
若 `u_accept < min(1, p_target[x]/q_draft[x])` 返回 `(x, True)`；否则从残差分布重采样返回 `(new_x, False)`。
（把随机数 `u_accept` 显式传入，便于确定性测试。）

In [ ]:
def accept_or_resample(p_target, q_draft, x, u_accept, rng):
    # TODO: 用 u_accept 与 min(1, p_target[x]/q_draft[x]) 比较决定接受
    #       拒绝时用 residual_dist(p_target, q_draft) 重采样
    #       返回 (token, 是否接受)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
pt = np.array([0.1, 0.6, 0.3])
qd = np.array([0.5, 0.2, 0.3])
# x=1: accept_prob = min(1, 0.6/0.2)=1 -> 任何 u 都接受
tok, acc = accept_or_resample(pt, qd, 1, 0.99, rng)
assert tok == 1 and acc is True, 'p>q 的 token 必接受'
# x=0: accept_prob = min(1, 0.1/0.5)=0.2 -> u=0.9 应拒绝
tok, acc = accept_or_resample(pt, qd, 0, 0.9, rng)
assert acc is False, 'u 大于接受概率应拒绝'
# 拒绝重采样必落在残差有质量处 (p>q: token1)
assert tok == 1, '残差只在 p>q 处(token1)有质量'
print('✅ 练习 1 通过：接受/拒绝单步正确')

## ✏️ 练习 2：期望确认 token 数闭式

实现 `expected_tokens(alpha, gamma)` 返回 `(1−α^(γ+1))/(1−α)`，并处理 `alpha==1` 的极限（应为 `gamma+1`）。
再实现 `best_gamma(alpha, c, gammas)`：在候选 γ 里选净加速最高的，返回 `(最优γ, 加速比)`。

In [ ]:
def expected_tokens(alpha, gamma):
    # TODO: 返回 (1-alpha**(gamma+1))/(1-alpha)；alpha==1 时返回 gamma+1
    raise NotImplementedError

def best_gamma(alpha, c, gammas=range(1, 11)):
    # TODO: 对每个 gamma 算 E/(gamma*c+1)，返回 (最优gamma, 加速比)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert abs(expected_tokens(0.5, 4) - (1-0.5**5)/(1-0.5)) < 1e-9
assert abs(expected_tokens(1.0, 4) - 5) < 1e-9, 'α=1 极限应为 γ+1'
assert expected_tokens(0.9, 8) > expected_tokens(0.5, 8), 'α 越高确认越多'
g, sp = best_gamma(0.8, 0.1)
assert 1 <= g <= 10 and sp > 1.0
# 与第 6 节的 net_speedup 对拍
assert abs(sp - max(net_speedup(0.8, gg, 0.1) for gg in range(1,11))) < 1e-9
print(f'✅ 练习 2 通过：期望 token 闭式 + 最优 γ={g} (加速 {sp:.2f}x)')

## ✏️ 练习 3：树形投机的期望确认数

一棵简化的 token 树：根下挂若干条**链**（每条链是一串候选 token），各链彼此独立、每个位置以接受率 `alpha` 被接受。
实现 `tree_expected_tokens(alpha, branch_lengths)`：返回「被接受的最长前缀长度 + 1」的期望。
提示：选最长被接受前缀 = 各链「连续接受前缀长度」的最大值；对每条长 L 的链，连续接受前缀长度 k 的分布是 `P(k)=α^k(1−α)`（k<L）、`P(L)=α^L`。

In [ ]:
def tree_expected_tokens(alpha, branch_lengths):
    # TODO: 对每条链算「连续接受前缀长度」的分布，求所有链该长度的最大值的期望，最后 +1
    #   做法：枚举 max_prefix = m (0..max(branch_lengths))，
    #         用 P(单链前缀>=m) 推 P(max>=m)，E[max]=sum_{m>=1} P(max>=m)，结果再 +1
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 单条链退化为链式投机：应等于 (1-α^(L+1))/(1-α)
for alpha, L in [(0.7, 4), (0.8, 5), (0.5, 3)]:
    got = tree_expected_tokens(alpha, [L])
    ref = (1 - alpha**(L+1)) / (1 - alpha)
    assert abs(got - ref) < 1e-9, f'单链应退化为链式: {got} vs {ref}'
# 多押几注(更多/更长的链)不会更差，通常更好
one = tree_expected_tokens(0.7, [4])
many = tree_expected_tokens(0.7, [4, 4, 4])
assert many >= one - 1e-9, '更多候选链确认数不应更少'
assert many > one, '通常严格更多(更可能命中更长前缀)'
print(f'✅ 练习 3 通过：单链={one:.3f}, 三链={many:.3f} —— 树形投机确认更长前缀')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def accept_or_resample(p_target, q_draft, x, u_accept, rng):
    accept_prob = min(1.0, p_target[x] / q_draft[x])
    if u_accept < accept_prob:
        return x, True
    resid = residual_dist(p_target, q_draft)
    return rng.choice(len(q_draft), p=resid), False

In [ ]:
# 练习 2 参考答案
def expected_tokens(alpha, gamma):
    if alpha >= 1.0:
        return gamma + 1
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)

def best_gamma(alpha, c, gammas=range(1, 11)):
    best = (0, 0.0)
    for g in gammas:
        sp = expected_tokens(alpha, g) / (g * c + 1)
        if sp > best[1]:
            best = (g, sp)
    return best

In [ ]:
# 练习 3 参考答案
def tree_expected_tokens(alpha, branch_lengths):
    # P(单条长L的链 连续接受前缀 >= m) = alpha**m  (m<=L), 0 if m>L
    def p_branch_ge(m, L):
        return alpha ** m if m <= L else 0.0
    Lmax = max(branch_lengths)
    E_max = 0.0
    for m in range(1, Lmax + 1):
        # P(max前缀 >= m) = 1 - ∏(1 - P(单链>=m))
        p_none = 1.0
        for L in branch_lengths:
            p_none *= (1 - p_branch_ge(m, L))
        E_max += (1 - p_none)
    return E_max + 1

---
## 🧪 真实数据胶囊：用真实接受率算端到端加速

下面是文献/实践中**真实报告**的接受率与 draft 速度（不同论文/模型对会有差异，这里取代表性区间）。
用它们算各方法的净加速，把公式接到现实。

> 这些数字随模型对、温度、领域而变；此处用作量级参考。EAGLE 因 draft 看到 target 特征，α 最高。

In [ ]:
# 真实报告的代表性数字（接受率 α，draft 单步相对 target 的成本 c）
REAL = {
    '小模型 draft (7B->70B)': dict(alpha=0.70, c=0.10),   # 独立小模型
    'Medusa (多头)':          dict(alpha=0.65, c=0.02),   # 头几乎免费但 α 中等
    'EAGLE (特征级)':         dict(alpha=0.80, c=0.05),   # α 最高
}

def best_speedup(alpha, c, gammas=range(1, 9)):
    return max(((1-alpha**(g+1))/(1-alpha)) / (g*c+1) for g in gammas)

print(f"{'方法':<22}{'α':>6}{'c':>6}{'最优加速':>9}")
for name, d in REAL.items():
    sp = best_speedup(d['alpha'], d['c'])
    print(f'{name:<22}{d["alpha"]:>6}{d["c"]:>6}{sp:>8.2f}x')
print('\n观察：低 c(Medusa/EAGLE 复用主干) + 高 α(EAGLE 特征级) 才能逼近 2-3x。')

**🧪 胶囊练习**：实现 `breakeven_alpha(c, gamma)`：在草稿长度 `gamma`、draft 成本 `c` 下，使净加速恰好 = 1（盈亏平衡）的接受率 α（数值扫描即可）。低于它就不值得投机。

In [ ]:
def breakeven_alpha(c, gamma):
    # TODO: 扫描 alpha in (0,1)，找使 net_speedup≈1 的最小 alpha
    #       net = ((1-a**(gamma+1))/(1-a)) / (gamma*c+1)
    raise NotImplementedError

In [ ]:
# 自测
a = breakeven_alpha(c=0.1, gamma=4)
assert 0.0 < a < 1.0
net = ((1-a**5)/(1-a)) / (4*0.1+1)
assert abs(net - 1.0) < 0.05, '盈亏平衡处净加速应≈1'
print(f'γ=4, c=0.1 的盈亏平衡接受率 α ≈ {a:.2f}（低于它投机反而变慢）')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def breakeven_alpha(c, gamma):
    for a in np.linspace(0.01, 0.99, 99):
        net = ((1 - a**(gamma+1))/(1 - a)) / (gamma*c + 1)
        if net >= 1.0:
            return float(a)
    return 1.0

---
### 小结
- decode **串行 + 带宽受限**：每次前向读一堆权重只产 1 token，算力闲置 → 投机解码顺便多产几个。
- **draft-then-verify**：draft 自回归猜 γ 个，target **一次前向并行验**，从左到右接受、遇拒即停 +1。
- **接受/拒绝规则** `min(1,p/q)` + 残差 `norm(max(0,p−q))` → 输出分布**精确等于 target（无损）**，与 draft 质量无关。
- 期望确认 `E=(1−α^(γ+1))/(1−α)`；净加速 `E/(γc+1)`，存在**最优 γ**。
- **Medusa/EAGLE** 免单独 draft、**树形投机**一次验多条路径，主线都是「让 draft 更像 target、把 α 推高」。

下一站：**模块 04 · Prefix Cache 与 PD 分离** —— 不重复 prefill、把两阶段拆到不同 GPU。